In [6]:
"""
Hybrid Detection Framework
──────────────────────────
Primary backbone  : YOLOv11  (high recall, learned features)
Validation layer  : Advanced OpenCV  (geometric precision filter)
 
Pipeline logic
──────────────
1. Run YOLO inference on the form image.
2. High-confidence predictions  (>= HIGH_CONF) → accepted directly.
3. Low-confidence predictions   (< HIGH_CONF, >= CONF_THRESH) →
      cross-referenced with OpenCV candidates via IoU overlap.
      • Overlap found  → accepted (YOLO box kept, validated by geometry).
      • No overlap     → subjected to strict geometric filtering before
                         acceptance; rejected if it fails.
4. OpenCV candidates with NO matching YOLO detection →
      added as supplementary detections (fills YOLO misses).
5. Final merged output is evaluated against ground-truth annotations.
"""
 
import os, json, math
from PIL import Image
import numpy as np
import cv2
from ultralytics import YOLO
 
# ── CONFIG ───────────────────────────────────────────────────────────────────
WEIGHTS_PATH   = "runs/detect/run_v26l/runs/train/form_detector/weights/best.pt"   # YOLOv11 weights
GT_FOLDER      = "clicking_mechanism/annotations"
IMAGE_FOLDER   = "data"
 
CONF_THRESH    = 0.25    # minimum YOLO confidence to consider at all
HIGH_CONF      = 0.60    # above this → accepted without OpenCV check
IOU_THRESHOLD  = 0.30    # minimum IoU to count as "confirmed overlap"
TOLERANCES     = [0.05, 0.10, 0.20]
 
CLASS_NAMES    = {0: "Checkboxes", 1: "Lines", 2: "Boxes"}
 
# OpenCV geometry thresholds (Advanced pipeline)
CHECKBOX_MIN      = 10
CHECKBOX_MAX      = 50
SQUARE_TOLERANCE  = 0.30
LINE_MIN_W        = 100
LINE_MAX_H        = 4
LABEL_SEARCH_PX   = 300
LABEL_PIXEL_RATIO = 0.01
# ─────────────────────────────────────────────────────────────────────────────
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  OPENCV PIPELINE  (Advanced)
# ═══════════════════════════════════════════════════════════════════════════════
 
def load_gray(path):
    img_pil = Image.open(path).convert("L")
    return np.array(img_pil)
 
def preprocess(gray):
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    return cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
 
def detect_checkboxes_cv(binary, img_w, img_h):
    contours, _ = cv2.findContours(binary, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    checkboxes = []
    for cnt in contours:
        peri   = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) != 4:
            continue
        x, y, w, h = cv2.boundingRect(approx)
        if w > img_w * 0.95 or h > img_h * 0.95:
            continue
        squareness = abs(1 - (w / h)) if h > 0 else 99
        if (CHECKBOX_MIN <= w <= CHECKBOX_MAX and
                CHECKBOX_MIN <= h <= CHECKBOX_MAX and
                squareness <= SQUARE_TOLERANCE):
            checkboxes.append((x, y, x + w, y + h))   # xyxy
    return checkboxes
 
def detect_lines_cv(binary, checkboxes_xyxy):
    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
    h_lines = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(h_lines, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        if w < LINE_MIN_W or h > LINE_MAX_H:
            continue
        lx1 = max(0, x - LABEL_SEARCH_PX)
        lx2 = max(0, x - 5)
        ly1 = max(0, y - 10)
        ly2 = min(binary.shape[0], y + h + 10)
        if lx2 <= lx1:
            continue
        region = binary[ly1:ly2, lx1:lx2]
        if region.size == 0 or np.count_nonzero(region) / region.size <= LABEL_PIXEL_RATIO:
            continue
        VICINITY = 20
        near_cb  = any(
            x >= cx1 - VICINITY and x + w <= cx2 + VICINITY and
            y >= cy1 - VICINITY and y + h <= cy2 + VICINITY
            for (cx1, cy1, cx2, cy2) in checkboxes_xyxy
        )
        if near_cb:
            continue
        lines.append((x, y, x + w, y + h))   # xyxy
    return lines
 
def detect_boxes_cv(binary, img_w, img_h, checkboxes_xyxy, lines_xyxy):
    """Wide rectangular regions that are neither checkboxes nor lines."""
    contours, _ = cv2.findContours(binary, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for cnt in contours:
        peri   = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) != 4:
            continue
        x, y, w, h = cv2.boundingRect(approx)
        if w > img_w * 0.95 or h > img_h * 0.95:
            continue
        # Exclude checkboxes
        squareness = abs(1 - (w / h)) if h > 0 else 99
        if (CHECKBOX_MIN <= w <= CHECKBOX_MAX and
                CHECKBOX_MIN <= h <= CHECKBOX_MAX and
                squareness <= SQUARE_TOLERANCE):
            continue
        # Must have meaningful height (not a thin line)
        if h <= LINE_MAX_H:
            continue
        # Must have reasonable box proportions
        aspect = w / h if h > 0 else 0
        if aspect < 1.2 or w < 40 or h < 8:
            continue
        boxes.append((x, y, x + w, y + h))   # xyxy
    return boxes
 
def run_opencv(image_path):
    """Run the full Advanced OpenCV pipeline; return dict of xyxy lists."""
    gray        = load_gray(image_path)
    h_img, w_img = gray.shape
    binary      = preprocess(gray)
    checkboxes  = detect_checkboxes_cv(binary, w_img, h_img)
    lines       = detect_lines_cv(binary, checkboxes)
    boxes       = detect_boxes_cv(binary, w_img, h_img, checkboxes, lines)
    return {0: checkboxes, 1: lines, 2: boxes}
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  GEOMETRY HELPERS
# ═══════════════════════════════════════════════════════════════════════════════
 
def iou(a, b):
    """Intersection over Union for two xyxy boxes."""
    ix1 = max(a[0], b[0]); iy1 = max(a[1], b[1])
    ix2 = min(a[2], b[2]); iy2 = min(a[3], b[3])
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    if inter == 0:
        return 0.0
    area_a = (a[2]-a[0]) * (a[3]-a[1])
    area_b = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (area_a + area_b - inter)
 
def best_iou(box, candidates):
    """Return (max_iou, best_candidate_index) against a list of boxes."""
    best, idx = 0.0, -1
    for i, c in enumerate(candidates):
        v = iou(box, c)
        if v > best:
            best, idx = v, i
    return best, idx
 
def passes_geometric_filter(box, cls_id):
    """
    Strict geometric sanity check for low-confidence YOLO boxes
    that have NO OpenCV counterpart.
    Returns True if the box looks plausible for its class.
    """
    x1, y1, x2, y2 = box
    w = x2 - x1;  h = y2 - y1
    if w <= 0 or h <= 0:
        return False
 
    if cls_id == 0:   # Checkbox: small and near-square
        aspect = w / h
        return (CHECKBOX_MIN <= w <= CHECKBOX_MAX and
                CHECKBOX_MIN <= h <= CHECKBOX_MAX and
                abs(1 - aspect) <= SQUARE_TOLERANCE)
 
    elif cls_id == 1:  # Line: wide and thin
        return w >= LINE_MIN_W and h <= LINE_MAX_H * 3   # slight slack for YOLO
 
    elif cls_id == 2:  # Box: rectangular, not too tiny
        aspect = w / h
        return w >= 40 and h >= 8 and aspect >= 1.2
 
    return False
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  HYBRID MERGE
# ═══════════════════════════════════════════════════════════════════════════════
 
def hybrid_merge(yolo_preds, opencv_preds):
    """
    yolo_preds  : {cls_id: [(x1,y1,x2,y2,conf), ...]}
    opencv_preds: {cls_id: [(x1,y1,x2,y2), ...]}
 
    Returns     : {cls_id: [(x1,y1,x2,y2), ...]}  — final merged detections
    """
    final = {cls: [] for cls in CLASS_NAMES}
 
    for cls_id in CLASS_NAMES:
        yolo_boxes  = yolo_preds.get(cls_id, [])
        cv_boxes    = list(opencv_preds.get(cls_id, []))   # mutable copy
        used_cv_idx = set()
 
        # ── Step 1 & 2: process every YOLO prediction ──────────────────────
        for (x1, y1, x2, y2, conf) in yolo_boxes:
            box = (x1, y1, x2, y2)
 
            if conf >= HIGH_CONF:
                # High-confidence → accept directly
                final[cls_id].append(box)
 
            else:
                # Low-confidence → cross-reference with OpenCV
                score, cv_idx = best_iou(box, cv_boxes)
 
                if score >= IOU_THRESHOLD:
                    # Confirmed by OpenCV geometry → accept YOLO box
                    final[cls_id].append(box)
                    used_cv_idx.add(cv_idx)
 
                else:
                    # No geometric confirmation → apply strict filter
                    if passes_geometric_filter(box, cls_id):
                        final[cls_id].append(box)
                    # else: silently drop — likely a false positive
 
        # ── Step 3: supplement with unmatched OpenCV candidates ────────────
        for i, cv_box in enumerate(cv_boxes):
            if i in used_cv_idx:
                continue
            # Only add if no already-accepted detection overlaps it
            overlap = any(iou(cv_box, accepted) >= IOU_THRESHOLD
                          for accepted in final[cls_id])
            if not overlap:
                final[cls_id].append(cv_box)
 
    return final
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  EVALUATION  (same tolerance-based matching as original scripts)
# ═══════════════════════════════════════════════════════════════════════════════
 
def load_gt(path, img_w, img_h):
    data = json.load(open(path))
    gt = {0: [], 1: [], 2: []}
 
    for cx, cy in data.get("checkboxes", []):
        s = 0.01 * min(img_w, img_h)
        gt[0].append((cx-s, cy-s, cx+s, cy+s))
 
    for x1, y1, x2, y2 in data.get("lines", []):
        pad_x = 0.01 * img_w; pad_y = 0.01 * img_h
        gt[1].append((min(x1,x2)-pad_x, min(y1,y2)-pad_y,
                      max(x1,x2)+pad_x, max(y1,y2)+pad_y))
 
    for x1, y1, x2, y2 in data.get("boxes", []):
        gt[2].append((x1, y1, x2, y2))
 
    return gt
 
def match_boxes(gt_boxes, pred_boxes, tol_x, tol_y):
    candidates = []
    for gi, (gx1,gy1,gx2,gy2) in enumerate(gt_boxes):
        for pi, (px1,py1,px2,py2) in enumerate(pred_boxes):
            if (abs(px1-gx1) <= tol_x and abs(py1-gy1) <= tol_y and
                    abs(px2-gx2) <= tol_x and abs(py2-gy2) <= tol_y):
                dist = (math.sqrt((gx1-px1)**2 + (gy1-py1)**2) +
                        math.sqrt((gx2-px2)**2 + (gy2-py2)**2))
                candidates.append((dist, gi, pi))
    candidates.sort(key=lambda x: x[0])
    matched_gt, matched_pred, count = set(), set(), 0
    for _, gi, pi in candidates:
        if gi not in matched_gt and pi not in matched_pred:
            matched_gt.add(gi); matched_pred.add(pi); count += 1
    return count
 
def compute_metrics(matches, total_pred, total_gt):
    p   = matches / total_pred if total_pred else 0
    r   = matches / total_gt   if total_gt   else 0
    f1  = (2*p*r / (p+r))      if (p+r)      else 0
    den = total_gt + total_pred - matches
    acc = matches / den         if den        else 0
    return p, r, f1, acc
 
 
# ═══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ═══════════════════════════════════════════════════════════════════════════════
 
def evaluate():
    model = YOLO(WEIGHTS_PATH)
 
    gt_stems  = {f[:-5] for f in os.listdir(GT_FOLDER)    if f.endswith(".json")}
    img_stems = {f[:-4]  for f in os.listdir(IMAGE_FOLDER) if f.endswith(".png")}
    stems     = sorted(gt_stems & img_stems)
 
    if not stems:
        print("No matching GT + image pairs found. Check your paths.")
        return
 
    for tol in TOLERANCES:
        totals = {cls: [0, 0, 0] for cls in CLASS_NAMES}   # [gt, pred, match]
 
        print("\n" + "=" * 60)
        print(f"TOLERANCE: {int(tol * 100)}%")
        print("=" * 60)
 
        for stem in stems:
            img_path = os.path.join(IMAGE_FOLDER, stem + ".png")
            gt_path  = os.path.join(GT_FOLDER,    stem + ".json")
            img_w, img_h = Image.open(img_path).size
            tol_x, tol_y = img_w * tol, img_h * tol
 
            # ── Ground truth ────────────────────────────────────────────────
            gt = load_gt(gt_path, img_w, img_h)
 
            # ── YOLO predictions ────────────────────────────────────────────
            results    = model.predict(img_path, conf=CONF_THRESH, verbose=False)[0]
            yolo_preds = {0: [], 1: [], 2: []}
            for box in results.boxes:
                cls_id = int(box.cls)
                if cls_id in yolo_preds:
                    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
                    conf            = float(box.conf)
                    yolo_preds[cls_id].append((x1, y1, x2, y2, conf))
 
            # ── OpenCV predictions ──────────────────────────────────────────
            opencv_preds = run_opencv(img_path)
 
            # ── Hybrid merge ────────────────────────────────────────────────
            merged = hybrid_merge(yolo_preds, opencv_preds)
 
            # ── Accumulate metrics ──────────────────────────────────────────
            for cls_id in CLASS_NAMES:
                matches = match_boxes(gt[cls_id], merged[cls_id], tol_x, tol_y)
                totals[cls_id][0] += len(gt[cls_id])
                totals[cls_id][1] += len(merged[cls_id])
                totals[cls_id][2] += matches
 
        for cls_id, name in CLASS_NAMES.items():
            gt_n, pred_n, match_n = totals[cls_id]
            p, r, f1, acc = compute_metrics(match_n, pred_n, gt_n)
            print(f"\n{name}:")
            print(f"  Precision: {p:.3f} | Recall: {r:.3f} | "
                  f"F1: {f1:.3f} | Accuracy (Jaccard): {acc:.3f}")
 
 
if __name__ == "__main__":
    evaluate()


TOLERANCE: 5%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.804 | Recall: 0.752 | F1: 0.777 | Accuracy (Jaccard): 0.636

Lines:
  Precision: 0.459 | Recall: 0.723 | F1: 0.562 | Accuracy (Jaccard): 0.390

Boxes:
  Precision: 0.305 | Recall: 0.673 | F1: 0.420 | Accuracy (Jaccard): 0.266

TOLERANCE: 10%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.812 | Recall: 0.759 | F1: 0.785 | Accuracy (Jaccard): 0.646

Lines:
  Precision: 0.468 | Recall: 0.737 | F1: 0.573 | Accuracy (Jaccard): 0.401

Boxes:
  Precision: 0.324 | Recall: 0.716 | F1: 0.446 | Accuracy (Jaccard): 0.287

TOLERANCE: 20%


libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: iCCP: known incorrect sRGB profile
libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
libpng warning: 


Checkboxes:
  Precision: 0.816 | Recall: 0.762 | F1: 0.788 | Accuracy (Jaccard): 0.650

Lines:
  Precision: 0.484 | Recall: 0.762 | F1: 0.592 | Accuracy (Jaccard): 0.420

Boxes:
  Precision: 0.373 | Recall: 0.824 | F1: 0.514 | Accuracy (Jaccard): 0.345


libpng warning: iCCP: profile 'ICC Profile': 0h: PCS illuminant is not D50
